# Configuring CometMirror
This notebook is all about configuring CometMirror. You will learn how to:
1) Specify time-dependent trajectories (e.g. current ramps, impurity injections)
2) Generate random walks for a subset of the parameters
3) Configure batches of simulation runs using `MultiCases` and `CombinatorialCases`

Let's begin by loading CometMirror for the SPARC PRD and print out the initial `State` and `Params`.

In [ ]:
import warnings
from pprint import pprint

from popsim.scenarios.sparc_prd.comet_mirror import build_comet_mirror_config

%load_ext autoreload
%autoreload 2

# Ignore the xarray warning about stripping away units.
warnings.filterwarnings("ignore", message="The unit of the quantity is stripped when downcasting to ndarray")

# Initialize the simulator.
model, state, params = build_comet_mirror_config()

pprint(state)
pprint(params)

## Simulating with State + Params

The `State` dataclass is vector of variables that are being simulated, while `Params` can be thought of as boundary conditions and/or assumptions. Letting $\mathbf{x}_t$ denote the state at time $t$ and $\mathbf{p}_t$ denote the params at time $t$, at every time step of the simulation, what essentially happens is something like this:
$$\mathbf{x}_{t+1} = f(\mathbf{x}_t, \mathbf{p}_t)$$


## The Magical Params Dataclass
The `Params` dataclass is where the magic happens, and where you get super-powers. Every element of it is configurable by you. Let's begin by defining a simulation time-base and defining a current ramp that starts 1 second into the simulation. While we're at it, why not define a auxiliary heating ramp rate, and also a tungsten impurity injection that occurs between (2.0, 2.1) seconds in the simulaiton?

In [ ]:
import dataclasses

import jax.numpy as jnp

from popsim.enums import Impurity
from popsim.simulators.comet_mirror.simulate import simulate

times = jnp.linspace(0, 5.0, 100)  # Simulate for 5 seconds with 100 steps.

# Create
new_params = dataclasses.replace(params)

# Manually define a current-ramp where the key is the time in seconds and the value is the current in Amperes.
new_params.plasma_current = {0.0: 8.7e6, 1.0: 8.7e6, 5.0: 4.0e6}

# Manually define an auxiliary heating power ramp where the key is the time in seconds and the value is the power in MW.
new_params.P_aux_MW = {0.0: 11.1, 5.0: 7.0}

# Manually define a quick tungsten spike. Note that under the hood linear interpolation is happening, so we need this
# perhaps somewhat awkward definition.
new_params.fueling19[Impurity.Tungsten] = {
    0.0: 0.0,
    1.999: 0.0,  # Start ramping impurities.
    2.0: 0.1,  # Impurity injection is now at 0.1.
    2.1: 0.1,  # Impurity holds at 0.1.
    2.10001: 0.0,  # Impurity drops back to 0.0.
    5.0: 0.0,  # Impurity holds at 0.0.
}

out = simulate(
    model=model,
    ts=times,
    initial_state=state,
    params=new_params,
)

## Directly Providing an Interpolated Function Instead
Well ain't that nifty?

But what if manually writing out times and stuff is a bit tedious? What if you want to have some other code that generates a sequence of times and values and you want to use that instead? Sure, why not. One go-to place is the `popsim.interp` module which we use below to specify a current ramp trajectory.

In [ ]:
from popsim.interp import interp

# Specify a ramp rate and create an array of plasma currents.
ramp_rate = -0.5e6
current_trajectory = 8.7e6 + ramp_rate * times

# Interpolate and apply the interpolated trajectory to the params struct.
new_params.plasma_current = interp(times, current_trajectory)

# Simulate the new trajectory.
out = simulate(
    model=model,
    ts=times,
    initial_state=state,
    params=new_params,
)

# Running Batches of Simulations with MultiCases

Okay, running one simulation is good, but running a gazillion is great!

Often times, we want to run a whole bunch of simulations with different settings. This is where `MultiCases` and `CombinatorialCases` come in.

As the names suggest, the former helps you specify multiple simulation cases, while the latter helps you automatically generate all possible combinations of simulation cases. Let's start with `MultiCases`. This is a good opportunity perhaps to introduce the random walk generator.

In [ ]:
import jax
from popsim.stochastic import generate_random_walks

diffusion_mags = {k: 0.15 for k in params.particle_confinement_scalar.keys()}
n_samps = 10
random_walks = generate_random_walks(
    jax.random.PRNGKey(42),
    n_samps,
    times,
    params.particle_confinement_scalar,
    diffusion_mags,
    return_interp=True,
)
pprint(random_walks)
# TODO: visualzie

You may recall from earlier that `particle_confinement_scalar` is a dictionary in the 

```python
      particle_confinement_scalar={<FuelSpecies.Tritium: -3>: 3.0,
                                    <FuelSpecies.Deuterium: -2>: 3.0,
                                    <Impurity.Helium: 2>: 10.0,
                                    <Impurity.Oxygen: 8>: 10.0,
                                    <Impurity.Tungsten: 74>: 10.0},
```
However, the output of the random walk generator is instead a list of `LinearInterpolation` things. But have no fear, we can still just wrap it in a `MultiCases` and it will work just fine.

Note that we can also use `MultiCases` with other variables, but we need to make sure that every `MultiCases` object has the same length or things will break. Just for the heck of it, let's also vary heating power across the random walks.

In [ ]:
from popsim.config_utils import MultiCases

new_params = dataclasses.replace(params)

# Apply the random walks to the params struct.
new_params.particle_confinement_scalar = MultiCases(cases=random_walks)

# Also vary heating across the cases. Note that we need to make sure all MultiCases have the same length.
heating_scales = jnp.linspace(0.8, 1.2, len(random_walks))
new_params.P_aux_MW = MultiCases(cases=[new_params.P_aux_MW * scale for scale in heating_scales])

out = simulate(
    model=model,
    ts=times,
    initial_state=state,
    params=new_params,
)

# Running Batches of Simulations with CombinatorialCases

Okay, great! So `MultiCases` allows us to specify multiple different scenarios. But sometimes, we want to generate all possible combinations of scenarios. For example, perhaps we want to scan auxiliary heating rates for every random walk trajectory we saw above. This is where `CombinatorialCases` comes in.

When you specify `CombinatorialCases`, every list gets combined with every otehr one in the `params`. So in the example below, the number of simulations run will be the product of the number of elements in each list (3 * 2 * 4 = 24). Oh yeah, you can also specify time-dependent trajectories in both `MultiCases` and `CombinatorialCases`!
```python
params.var0 = CombinatorialCases([1, 2, 3])
params.var1 = CombinatorialCases([{0.0: 1.0, 1.0: 2.0}, {0.0: 1.0, 1.0: 3.0}]) # Specifying different time dependent trajectories
params.var2 = CombinatorialCases([500, 200, 100, 50])
```